# Results Analysis

Load and analyze evaluation results from experiments.
- Compare scheduler performance across benchmarks
- DAG structure vs accuracy analysis
- Search progress visualization
- Generate paper figures

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['font.size'] = 11

RESULTS_DIR = Path('../results')
FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

print('Imports OK')

## 1. Load Results

In [ ]:
# Load summary from eval_dags.py output
# Change path as needed
summary_path = RESULTS_DIR / 'llada_dag_eval' / 'summary.json'

if summary_path.exists():
    with open(summary_path) as f:
        results = json.load(f)
    print('Loaded results for:')
    for dag_name, benchmarks in results.items():
        print(f'  DAG: {dag_name} | benchmarks: {list(benchmarks.keys())}')
else:
    # Demo with synthetic data
    print('No results found. Using synthetic demo data.')
    results = {
        'confidence': {'mbpp': {'pass@1': 0.31}, 'humaneval': {'pass@1': 0.24}, 'hotpotqa': {'exact_match': 0.38}, 'mmlu': {'accuracy': 0.52}},
        'random':     {'mbpp': {'pass@1': 0.21}, 'humaneval': {'pass@1': 0.17}, 'hotpotqa': {'exact_match': 0.29}, 'mmlu': {'accuracy': 0.41}},
        'linear':     {'mbpp': {'pass@1': 0.28}, 'humaneval': {'pass@1': 0.22}, 'hotpotqa': {'exact_match': 0.35}, 'mmlu': {'accuracy': 0.49}},
        'cot':        {'mbpp': {'pass@1': 0.36}, 'humaneval': {'pass@1': 0.29}, 'hotpotqa': {'exact_match': 0.43}, 'mmlu': {'accuracy': 0.55}},
        'skeleton':   {'mbpp': {'pass@1': 0.33}, 'humaneval': {'pass@1': 0.26}, 'hotpotqa': {'exact_match': 0.40}, 'mmlu': {'accuracy': 0.53}},
        'bidirectional': {'mbpp': {'pass@1': 0.30}, 'humaneval': {'pass@1': 0.24}, 'hotpotqa': {'exact_match': 0.38}, 'mmlu': {'accuracy': 0.51}},
    }

## 2. Main Results Table

In [ ]:
def get_metric(result, benchmark):
    if not result:
        return 0.0
    for key in ['pass@1', 'accuracy', 'exact_match', 'f1']:
        if key in result:
            return result[key]
    return 0.0

dag_names = list(results.keys())
benchmarks = list(next(iter(results.values())).keys())

# Build matrix
matrix = np.zeros((len(dag_names), len(benchmarks)))
for i, dag in enumerate(dag_names):
    for j, bm in enumerate(benchmarks):
        matrix[i, j] = get_metric(results[dag].get(bm, {}), bm)

# Print table
print(f"{'DAG':<20}" + ''.join(f'{bm:>14}' for bm in benchmarks))
print('-' * (20 + 14 * len(benchmarks)))
for i, dag in enumerate(dag_names):
    row = f'{dag:<20}' + ''.join(f'{matrix[i,j]:>14.4f}' for j in range(len(benchmarks)))
    print(row)

## 3. Bar Chart Comparison

In [ ]:
fig, axes = plt.subplots(1, len(benchmarks), figsize=(4 * len(benchmarks), 5), sharey=False)

colors = plt.cm.Set2(np.linspace(0, 1, len(dag_names)))

for j, (ax, bm) in enumerate(zip(axes, benchmarks)):
    values = matrix[:, j]
    bars = ax.bar(range(len(dag_names)), values, color=colors, alpha=0.85)
    ax.set_xticks(range(len(dag_names)))
    ax.set_xticklabels(dag_names, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('Score')
    ax.set_title(bm.upper())
    ax.set_ylim(0, min(1.0, values.max() * 1.2))
    ax.grid(True, alpha=0.3, axis='y')
    # Annotate bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('LLaDA + DAG Unmasking: Performance by Benchmark', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'main_results_bar.pdf', bbox_inches='tight')
plt.show()
print('Saved main_results_bar.pdf')

## 4. DAG Structure vs Performance

In [ ]:
from dllm_reason.graph.dag import TokenDAG
from dllm_reason.graph.templates import (
    chain_of_thought_dag, skeleton_then_detail_dag, bidirectional_dag
)
from dllm_reason.eval.dag_analysis import analyze_dag, plot_dag_stats_vs_performance

SEQ_LEN = 256

dag_objects = {
    'confidence': TokenDAG.no_edges(SEQ_LEN),
    'random':     TokenDAG.no_edges(SEQ_LEN),
    'linear':     TokenDAG.linear_chain(SEQ_LEN),
    'cot':        chain_of_thought_dag(SEQ_LEN, 4),
    'skeleton':   skeleton_then_detail_dag(SEQ_LEN,
                      list(range(0, SEQ_LEN, 3)), list(range(1, SEQ_LEN, 3))),
    'bidirectional': bidirectional_dag(SEQ_LEN, 4),
}

# Average performance across benchmarks
avg_scores = {dag: matrix[i].mean() for i, dag in enumerate(dag_names)}
print('Average scores:', avg_scores)

stats_list = [analyze_dag(dag_objects[n]) for n in dag_names if n in dag_objects]
scores = [avg_scores[n] for n in dag_names if n in dag_objects]
names = [n for n in dag_names if n in dag_objects]

fig = plot_dag_stats_vs_performance(stats_list, scores, names, figsize=(14, 8))
plt.savefig(FIGURES_DIR / 'dag_structure_vs_accuracy.pdf', bbox_inches='tight')
plt.show()

## 5. Search History (if available)

In [ ]:
search_result_path = RESULTS_DIR / 'dag_search' / 'search_result.json'

if search_result_path.exists():
    with open(search_result_path) as f:
        search_result = json.load(f)
    
    from dllm_reason.eval.dag_analysis import search_history_plot
    fig = search_history_plot(
        search_result['history'],
        title=f"{search_result['method']} DAG Search on {search_result['dataset']}"
    )
    plt.savefig(FIGURES_DIR / 'search_history.pdf', bbox_inches='tight')
    plt.show()
    
    print(f"Best fitness: {search_result['best_fitness']:.4f}")
    print(f"DAG stats: {search_result['dag_stats']}")
else:
    print('No search results found. Run scripts/search_dag.py first.')

## 6. Unmasking Timeline Paper Figure

In [ ]:
# Generate the key figure: unmasking timeline for each DAG
# This is typically Figure 2 or 3 in the paper

SEQ_LEN_VIZ = 64
NUM_STEPS = 16

viz_dags = {
    'Random': TokenDAG.no_edges(SEQ_LEN_VIZ),
    'Left-to-Right': TokenDAG.linear_chain(SEQ_LEN_VIZ),
    'Chain-of-Thought': chain_of_thought_dag(SEQ_LEN_VIZ, 4),
    'Skeleton-Detail': skeleton_then_detail_dag(
        SEQ_LEN_VIZ,
        list(range(0, SEQ_LEN_VIZ, 3)),
        list(range(1, SEQ_LEN_VIZ, 3))
    ),
}

fig, axes = plt.subplots(len(viz_dags), 1, figsize=(14, 6))
for ax, (name, dag) in zip(axes, viz_dags.items()):
    schedule = dag.to_mask_schedule(NUM_STEPS)
    step_map = torch.full((SEQ_LEN_VIZ,), -1, dtype=torch.float)
    for step, positions in enumerate(schedule):
        for pos in positions:
            if pos < SEQ_LEN_VIZ and step_map[pos] == -1:
                step_map[pos] = step
    
    im = ax.imshow(step_map.unsqueeze(0).numpy(), aspect='auto',
                   cmap='plasma', vmin=0, vmax=NUM_STEPS - 1,
                   interpolation='nearest')
    ax.set_yticks([])
    ax.set_ylabel(name, rotation=0, labelpad=90, va='center')
    ax.set_xticks([])

axes[-1].set_xlabel('Token Position →', fontsize=10)
fig.colorbar(im, ax=axes, label='Unmasking Step', fraction=0.02)
fig.suptitle('Unmasking Order by DAG Structure', fontsize=13, fontweight='bold')
plt.savefig(FIGURES_DIR / 'unmasking_timeline.pdf', bbox_inches='tight')
plt.show()
print('Saved unmasking_timeline.pdf')